<a href="https://colab.research.google.com/github/26200602/SIA-Agentic-AI-Architecture/blob/main/use-cases/shadow-diagnostic-engine/poc/sia_fsm_boundary_sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Environment Setup & Data Contract Loading
import json
import uuid
import time
from datetime import datetime, timezone
import jsonschema
from jsonschema import validate, ValidationError

# FSM Decision Packet Schema (Draft-07 Specification)
DECISION_PACKET_SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "FSM Decision Packet Schema",
    "description": "Data contract for FSM decision packets under the SIA Framework. Curated by MSK.",
    "type": "object",
    "properties": {
        "packet_id": {
            "type": "string",
            "format": "uuid"
        },
        "timestamp": {
            "type": "string",
            "format": "date-time"
        },
        "fsm_state_id": {
            "type": "string",
            "enum": [
                "FSM_ERR_OVERRIDE_001",
                "FSM_STATE_NORMAL",
                "FSM_STATE_TRANSITIONING",
                "FSM_STATE_UNKNOWN"
            ]
        },
        "context_latency_ms": {
            "type": "integer",
            "minimum": 0
        },
        "data_sovereignty": {
            "type": "object",
            "properties": {
                "transient_flush_verified": {"type": "boolean"},
                "raw_text_retained": {"type": "boolean", "const": False}
            },
            "required": ["transient_flush_verified", "raw_text_retained"],
            "additionalProperties": False
        }
    },
    "required": [
        "packet_id",
        "timestamp",
        "fsm_state_id",
        "context_latency_ms",
        "data_sovereignty"
    ],
    "additionalProperties": False
}

print("✅ Environment initialized. FSM Decision Packet Schema loaded.")

✅ Environment initialized. FSM Decision Packet Schema loaded.


In [4]:
# Cell 2: Ephemeral SLM Ingestion & Transient Flushing Simulation
def process_slm_input(raw_jargon_text: str) -> dict:
    start_time = time.time()

    # Simulated SLM Context Alignment (Mapping front-line jargon to deterministic FSM State)
    jargon_mapping = {
        "跟口頭指示做住先": "FSM_ERR_OVERRIDE_001",
        "系統正常運行中": "FSM_STATE_NORMAL",
        "狀態切換中": "FSM_STATE_TRANSITIONING"
    }

    # Default mapping fallback
    fsm_state = jargon_mapping.get(raw_jargon_text, "FSM_STATE_UNKNOWN")

    # Measure latency
    latency_ms = int((time.time() - start_time) * 1000) + 120  # Simulated inference overhead

    # Data Sovereignty Enforcement: Transient Flushing (Purging Raw Text Memory)
    del raw_jargon_text
    transient_flush_verified = True
    raw_text_retained = False

    # Construct Decision Packet
    packet = {
        "packet_id": str(uuid.uuid4()),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "fsm_state_id": fsm_state,
        "context_latency_ms": latency_ms,
        "data_sovereignty": {
            "transient_flush_verified": transient_flush_verified,
            "raw_text_retained": raw_text_retained
        }
    }
    return packet


# Execute with test input
input_text = "跟口頭指示做住先"
decision_packet = process_slm_input(input_text)

print("✅ SLM Processing Complete. Generated Packet:")
print(json.dumps(decision_packet, indent=2))

✅ SLM Processing Complete. Generated Packet:
{
  "packet_id": "d454e207-6238-4f68-a4e8-669960963060",
  "timestamp": "2026-08-17T07:56:58.532760+00:00",
  "fsm_state_id": "FSM_ERR_OVERRIDE_001",
  "context_latency_ms": 120,
  "data_sovereignty": {
    "transient_flush_verified": true,
    "raw_text_retained": false
  }
}


In [5]:
# Cell 3: Data Contract Validation & Compliance Audit
try:
    # Validate decision packet against Draft-07 JSON Schema loaded in Cell 1
    validate(instance=decision_packet, schema=DECISION_PACKET_SCHEMA)
    print("✅ Schema Validation PASSED: Packet complies with decision-packet.schema.json")
    print(f"✅ Zero-Knowledge Sovereignty PASSED: raw_text_retained = {decision_packet['data_sovereignty']['raw_text_retained']}")
    print(f"✅ Transient Flush Verified: {decision_packet['data_sovereignty']['transient_flush_verified']}")
except ValidationError as e:
    print(f"❌ Schema Validation FAILED: {e.message}")

✅ Schema Validation PASSED: Packet complies with decision-packet.schema.json
✅ Zero-Knowledge Sovereignty PASSED: raw_text_retained = False
✅ Transient Flush Verified: True
